# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Phân rã nhiệm vụ

Phân rã nhiệm vụ là cốt lõi của mẫu thiết kế lập kế hoạch. Thay vì yêu cầu một tác nhân duy nhất
xử lý một yêu cầu phức tạp từ đầu đến cuối, chúng ta chia vấn đề thành các **phần nhiệm vụ** nhỏ hơn, được định nghĩa rõ ràng.
Mỗi phần nhiệm vụ được giao cho một tác nhân chuyên môn (ví dụ, chuyến bay, khách sạn, hoạt động) với các
ưu tiên và thứ tự phụ thuộc rõ ràng.

Cách tiếp cận này mang lại một số lợi ích:
- **Rõ ràng**: mỗi phần nhiệm vụ có một trách nhiệm duy nhất.
- **Tính song song**: các phần nhiệm vụ độc lập có thể chạy đồng thời.
- **Độ tin cậy**: các lỗi được cô lập trong các phần nhiệm vụ riêng lẻ.
- **Theo dõi ngân sách**: chi phí được ước tính cho từng phần nhiệm vụ và tổng hợp lại.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Tạo một Đại lý Lập kế hoạch với Đầu ra Có cấu trúc

Đại lý lập kế hoạch hoạt động như một **điều phối viên tiếp tân**. Khi nhận yêu cầu du lịch cấp cao, nó
tạo ra một `TravelPlan` có cấu trúc — phân rã yêu cầu thành các nhiệm vụ phụ, thiết lập thứ tự ưu tiên,
và xác định các mối phụ thuộc để một nhân viên phục vụ hay tầng thực thi có thể tiến hành công việc.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Thực hiện Kế hoạch với Các Công cụ Chuyên gia

Khi nhân viên lễ tân đã tạo ra một kế hoạch có cấu trúc, **đại lý phục vụ** sẽ thực hiện nó.
Mỗi công cụ chuyên gia xử lý một loại công việc phụ (chuyến bay, khách sạn, hoạt động). Đại lý phục vụ
sẽ lặp qua các công việc phụ trong kế hoạch theo thứ tự phụ thuộc và gửi từng công việc đến
công cụ phù hợp.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Tóm tắt

Trong bài học này, bạn đã học được **Mẫu Thiết kế Lập kế hoạch** cho các tác nhân AI:

1. **Phân rã Nhiệm vụ** — Một tác nhân lập kế hoạch ở quầy lễ tân chia yêu cầu du lịch phức tạp thành
   các nhiệm vụ con có cấu trúc bằng cách sử dụng các mô hình Pydantic, phân công mỗi nhiệm vụ cho một tác nhân chuyên môn với các ưu tiên
   và phụ thuộc.
2. **Đầu ra Có cấu trúc** — Bằng cách truyền `response_format`, tác nhân trả về một đối tượng
   `TravelPlan` được xác thực thay vì văn bản tự do, giúp xử lý tiếp theo trở nên đáng tin cậy.
3. **Thực thi Kế hoạch** — Một tác nhân lễ tân lặp qua các nhiệm vụ con sử dụng các công cụ chuyên môn
   (`book_flight`, `reserve_hotel`, `book_activity`) để thực hiện kế hoạch và báo cáo kết quả.

Mẫu này tách biệt *việc cần làm* (lập kế hoạch) khỏi *cách thực hiện* (thực thi), giúp các tác nhân
trở nên mô-đun hơn, dễ kiểm thử và dễ mở rộng hơn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
